# Team Workbook — Day 3
## Predictive Analytics I  ·  Unit III

Descriptive told you *what happened*. Diagnostic told you *why*, sort of. Today you build something
that says *what will happen* for a row it has never seen.

**The one idea that separates today from every other day:**

> A model is only worth anything on data it has never seen. Every score you report today comes from
> rows the model was not allowed to look at while it was learning.

---


## Step 0 — Setup

Load the file you saved yesterday. If you did not get that far, the raw file works too.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)

FILE     = "day2_clean.csv"     # yesterday's output; use your raw file if you must
NUM_COL  = "____"               # your measure
CAT_COL  = "____"               # your category

# Leave TARGET as None and a target will be DERIVED from NUM_COL.
# Set it to a column name only if your data has a real thing worth predicting.
TARGET   = None
TASK     = "auto"               # "auto" | "regression" | "classification"

df = pd.read_csv(FILE)
df.columns = df.columns.str.strip()
df[NUM_COL] = pd.to_numeric(df[NUM_COL], errors="coerce")
print(df.shape); df.head()


### Paste the modelling kit

Run once. These helpers are what let one notebook fit models on every team's data.


In [ ]:
"""
MLKIT  —  the modelling helpers used on Days 3, 4 and 5.
Written so that the SAME code works on every team's dataset.
Paste this whole cell once, near the top of your notebook.
"""
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

MAX_ONEHOT = 25          # a category with more levels than this is dropped
MAX_NULL_FRAC = 0.5      # a column emptier than this is dropped


# ---------------------------------------------------------------- target
def make_target(df, num_col=None, target=None, task="auto", verbose=True):
    """
    Returns (y, task, derived_from, label).
    target=None  -> derive a yes/no target by splitting num_col at its median.
                    This works on ANY dataset that has one number in it.
    """
    derived_from = None
    if target is None:
        if num_col is None:
            raise ValueError("Give either target= or num_col= so a target can be derived.")
        s = pd.to_numeric(df[num_col], errors="coerce")
        cut = s.median()
        y = (s >= cut).astype("float")
        y[s.isna()] = np.nan
        task, derived_from = "classification", num_col
        label = f"is {num_col} at or above its median ({cut:,.2f})?"
    else:
        s = df[target]
        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)
            if task == "auto":
                task = "classification" if nunique <= 10 else "regression"
            y = pd.to_numeric(s, errors="coerce")
        else:
            task = "classification" if task == "auto" else task
            levels = sorted(s.dropna().astype(str).unique())
            if len(levels) == 2:
                mapping = {levels[0]: 0, levels[1]: 1}
                y = s.astype(str).map(mapping)
                y[s.isna()] = np.nan
                print(f"ENCODED  : {levels[0]} -> 0    {levels[1]} -> 1  "
                      f"(so \"positive\" below means {levels[1]})")
            else:
                y = s.astype(str)
        label = f"{target}"

    if verbose:
        print(f"TARGET   : {label}")
        print(f"TASK     : {task}")
        if derived_from:
            print(f"           (derived from {derived_from} - so {derived_from} MUST be excluded from the features)")
        if task == "classification":
            vc = pd.Series(y).value_counts(dropna=True)
            share = (vc / vc.sum() * 100).round(1)
            print(f"BALANCE  : " + "  ".join(f"{k}={v}%" for k, v in share.items()))
            if share.max() > 80:
                print("           WARNING - one class dominates. Accuracy will look great and mean nothing.")
    return y, task, derived_from, label


# ---------------------------------------------------------------- features
def make_features(df, y, derived_from=None, target=None, drop=None, verbose=True):
    """One-hot encodes, imputes, and removes anything that would leak or be useless."""
    drop = set(drop or [])
    if derived_from: drop.add(derived_from)
    if target:       drop.add(target)

    X = df.copy()
    n = len(X)
    removed = []

    for c in list(X.columns):
        if c in drop:
            removed.append((c, "excluded on purpose - it would leak the answer" if c == derived_from
                               else "this is the target")); X = X.drop(columns=c); continue
        s = X[c]
        if s.isnull().mean() > MAX_NULL_FRAC:
            removed.append((c, f"{s.isnull().mean()*100:.0f}% empty")); X = X.drop(columns=c); continue
        if s.nunique(dropna=True) <= 1:
            removed.append((c, "same value in every row")); X = X.drop(columns=c); continue
        if pd.api.types.is_datetime64_any_dtype(s):
            removed.append((c, "a date - engineer month/weekday from it first")); X = X.drop(columns=c); continue
        if not pd.api.types.is_numeric_dtype(s):
            u = s.nunique(dropna=True)
            if u == n:
                looks_dated = pd.to_datetime(s, errors="coerce", format="mixed").notna().mean() > 0.8
                removed.append((c, "a date - engineer month/weekday from it first" if looks_dated
                                   else "one value per row - an ID, not a feature"))
                X = X.drop(columns=c); continue
            if u > MAX_ONEHOT:
                removed.append((c, f"{u} categories - too many to one-hot")); X = X.drop(columns=c); continue
        else:
            if pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) == n and n > 10:
                removed.append((c, "one value per row - an ID, not a feature")); X = X.drop(columns=c); continue

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    for c in num_cols:
        if X[c].isnull().any():
            X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).str.strip().fillna("missing")

    X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=float)

    keep = pd.Series(y).notna().values
    X, y = X.loc[keep], pd.Series(y).loc[keep]

    if verbose:
        print(f"\nFEATURES : {X.shape[1]} columns built from {len(num_cols)} numeric + {len(cat_cols)} categorical")
        print(f"ROWS     : {len(X)} usable")
        if removed:
            print("REMOVED  :")
            for c, why in removed[:12]:
                print(f"           {c:<24} {why}")
    return X, y


# ---------------------------------------------------------------- split
def split(X, y, task, test_size=0.25, seed=42):
    strat = y if (task == "classification" and pd.Series(y).value_counts().min() >= 2) else None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=strat)
    print(f"TRAIN    : {len(Xtr)} rows      TEST: {len(Xte)} rows  (the test rows are never used to fit)")
    return Xtr, Xte, ytr, yte


# ---------------------------------------------------------------- metrics
def score(y_true, y_pred, task, y_prob=None):
    if task == "regression":
        return {"MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
                "R2": r2_score(y_true, y_pred)}
    avg = "binary" if pd.Series(y_true).nunique() == 2 else "macro"
    out = {"Accuracy": accuracy_score(y_true, y_pred),
           "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
           "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
           "F1": f1_score(y_true, y_pred, average=avg, zero_division=0)}
    if y_prob is not None and pd.Series(y_true).nunique() == 2:
        try: out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        except Exception: pass
    return out


RESULTS = {}

def evaluate(name, model, Xtr, Xte, ytr, yte, task, store=True, verbose=True):
    """Fit, predict on the held-out test set, score, remember the result."""
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = None
    if task == "classification" and hasattr(model, "predict_proba"):
        try: prob = model.predict_proba(Xte)[:, 1]
        except Exception: pass
    s = score(yte, pred, task, prob)
    if store: RESULTS[name] = s
    if verbose:
        print(f"{name:<22} " + "   ".join(f"{k} {v:.3f}" for k, v in s.items()))
    return model, s


def baseline(Xtr, Xte, ytr, yte, task):
    """The score to beat. A model that cannot beat this has learnt nothing."""
    m = DummyRegressor(strategy="mean") if task == "regression" else DummyClassifier(strategy="most_frequent")
    _, s = evaluate("BASELINE (no model)", m, Xtr, Xte, ytr, yte, task)
    return s


def leaderboard():
    if not RESULTS:
        print("Nothing evaluated yet."); return None
    t = pd.DataFrame(RESULTS).T.round(3)
    key = "R2" if "R2" in t.columns else "F1"
    return t.sort_values(key, ascending=False)


def cv(model, X, y, task, folds=5):
    scoring = "r2" if task == "regression" else "f1_macro"
    kf = KFold(folds, shuffle=True, random_state=42) if task == "regression" \
         else StratifiedKFold(folds, shuffle=True, random_state=42)
    s = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    print(f"{folds}-fold {scoring}: " + " ".join(f"{v:.3f}" for v in s))
    print(f"  mean {s.mean():.3f}   spread +/- {s.std():.3f}")
    return s


def leak_check(X, y, task, thresh=0.95):
    """
    Flags any single feature that predicts the target almost perfectly on its own.
    The automatic guard only removes the column the target was DERIVED from.
    It cannot know that m1..m5 add up to total - only you can. This finds those.
    """
    yy = pd.Series(y).astype(float) if task == "regression" else pd.Series(y)
    suspects = []
    for c in X.columns:
        col = X[c]
        if col.nunique() <= 1:
            continue
        if task == "regression":
            r = abs(np.corrcoef(col, yy)[0, 1])
        else:
            try:    r = abs(np.corrcoef(col, pd.factorize(yy)[0])[0, 1])
            except Exception: continue
        if np.isfinite(r) and r >= thresh:
            suspects.append((c, round(float(r), 3)))
    if suspects:
        print("POSSIBLE LEAK - these features almost ARE the target:")
        for c, r in sorted(suspects, key=lambda t: -t[1]):
            print(f"   {c:<28} correlation {r}")
        print("   If a feature is computed FROM the target, drop it. A model that")
        print("   sees the answer is not predicting - it is reading.")
    else:
        print("No single feature gives the answer away. (Combinations still might.)")
    return suspects


def importances(model, X, top=12):
    if hasattr(model, "feature_importances_"):
        v = pd.Series(model.feature_importances_, index=X.columns)
    elif hasattr(model, "coef_"):
        c = np.ravel(model.coef_)
        v = pd.Series(np.abs(c), index=X.columns)
    else:
        print("This model does not expose importances."); return None
    return v.sort_values(ascending=False).head(top).round(4)


---
## Step 1 — What are you actually predicting?

Two kinds of question, and the kind decides everything after it:

| | Question | Target looks like | Example |
|---|---|---|---|
| **Regression** | *how much?* | a number with many values | predict the order value |
| **Classification** | *which one?* | a label, or yes/no | predict whether the order is large |

**If your dataset has no obvious thing to predict — which is most of them — you derive one.**
Split your measure at its median and ask *"is this row above average?"* That is a real, honest,
answerable prediction problem, and it works on any dataset with a single number in it.


In [ ]:
y, task, derived_from, label = make_target(df, num_col=NUM_COL, target=TARGET, task=TASK)


> **If a WARNING about one class dominating appeared**, read it carefully. With a 90/10 split, a
> model that always guesses the big class scores 90% accuracy and is completely useless. Step 5
> proves that to you.


---
## Step 2 — Features: what the model is allowed to look at

`make_features` one-hot encodes your categories, fills gaps, and throws out anything that cannot help
or would cheat. **Read the REMOVED list** — it is the most informative output in this notebook.


In [ ]:
X, y = make_features(df, y, derived_from=derived_from, target=TARGET)
X.head()


The line that says *"excluded on purpose - it would leak the answer"* is the important one. If your
target is *"is `amount` above its median?"*, then `amount` itself must not be a feature — a model
given `amount` scores 100% and has learnt nothing at all. That is **leakage**, and it is the single
most common way a student model looks brilliant and is worthless.


### Step 2b — The leak the computer cannot find for you

The automatic guard only removes the column your target was *derived from*. It cannot know that
`m1 + m2 + m3 + m4 + m5` adds up to `total`, or that `discount_amount` is computed from `price`.
Only you know that. This checks for the obvious cases.


In [ ]:
suspects = leak_check(X, y, task)


If something is flagged, ask one question: **would I know this value at the moment I need the
prediction?** If the answer is no, drop it:

```python
X = X.drop(columns=["the_leaky_column"])
```


---
## Step 3 — Train and test: the whole discipline in one line

You hide a quarter of your rows, fit the model on the rest, and score it on the hidden quarter.
That is it. That is what makes a score believable.


In [ ]:
X_train, X_test, y_train, y_test = split(X, y, task, test_size=0.25, seed=42)


> **Why 42?** It is just a number that makes the random split reproducible, so your results do not
> change every time you re-run. Any fixed number works. What matters is that you fix one.


---
## Step 4 — The baseline: the score your model must beat

Before any model, find out how well you do with **no model at all** — always guess the average
(regression) or always guess the commonest class (classification).

Skipping this step is how people end up proud of a model that is worse than guessing.


In [ ]:
base = baseline(X_train, X_test, y_train, y_test, task)


**Write this number down.** Every model below gets compared to it, not to 100%.


---
## Step 5 — Your first model


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

if task == "regression":
    model, s = evaluate("LinearRegression", LinearRegression(),
                        X_train, X_test, y_train, y_test, task)
else:
    model, s = evaluate("LogisticRegression", LogisticRegression(max_iter=3000),
                        X_train, X_test, y_train, y_test, task)


### What the numbers mean

**If you are doing regression:**

| Metric | Plain meaning | Good is |
|---|---|---|
| **MAE** | on average, how far off you are, in your own units | as small as possible |
| **RMSE** | same, but punishes big misses much harder | close to MAE = few big misses |
| **R²** | share of the variation you explained. 0 = no better than the average. | above 0 and rising |

A **negative R²** means your model is worse than always guessing the mean. It happens, it is not a
crash, and it is information.

**If you are doing classification:**

| Metric | Plain meaning | Care about it when |
|---|---|---|
| **Accuracy** | share you got right | classes are balanced — and only then |
| **Precision** | of the ones you flagged, how many were real | a false alarm is expensive |
| **Recall** | of the real ones, how many you caught | a miss is expensive |
| **F1** | the balance of the two | you need one number |
| **ROC AUC** | how well you rank, at any threshold | comparing models |

> **Precision or recall — pick one before you look.** Screening for a disease? Recall: missing a case
> is far worse than a false alarm. Sending an expensive intervention? Precision: wasting it hurts.


### Step 5b — Look inside the model


In [ ]:
if task == "regression":
    coefs = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
    print("Each feature's effect on the prediction, per unit:\n")
    print(coefs.head(12).round(4).to_string())
    print(f"\nintercept (the prediction when everything is zero): {model.intercept_:,.3f}")
else:
    coefs = pd.Series(np.ravel(model.coef_), index=X.columns).sort_values(key=abs, ascending=False)
    print("Which features push the answer toward 1, and which toward 0:\n")
    print(coefs.head(12).round(4).to_string())

top = coefs.abs().sort_values(ascending=False).head(10).index
plt.figure(figsize=(7, 3.6))
sns.barplot(x=coefs[top].values, y=list(top), color="#2194D4")
plt.axvline(0, color="#6B7280", lw=1)
plt.title("Biggest effects (sign matters)"); plt.tight_layout(); plt.show()


**A large coefficient is not proof of importance.** A feature measured in rupees and a feature
measured in percent are not comparable until you scale them — which is tomorrow morning's first job.


---
## Step 6 — Check the mistakes, not just the score


In [ ]:
pred = model.predict(X_test)

if task == "regression":
    resid = y_test - pred
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].scatter(pred, y_test, s=16, alpha=.6, color="#2194D4")
    lims = [min(pred.min(), y_test.min()), max(pred.max(), y_test.max())]
    ax[0].plot(lims, lims, "--", color="#B4342F")
    ax[0].set_xlabel("predicted"); ax[0].set_ylabel("actual"); ax[0].set_title("Predicted vs actual")

    ax[1].scatter(pred, resid, s=16, alpha=.6, color="#2194D4")
    ax[1].axhline(0, color="#B4342F", ls="--")
    ax[1].set_xlabel("predicted"); ax[1].set_ylabel("error"); ax[1].set_title("Where the errors are")
    plt.tight_layout(); plt.show()
    print("A good residual plot is a shapeless cloud around zero.")
    print("A funnel or a curve means the model is systematically wrong somewhere.")
else:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    cm = confusion_matrix(y_test, pred)
    fig, ax = plt.subplots(figsize=(4.4, 3.8))
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion matrix"); plt.tight_layout(); plt.show()
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        print(f"correctly caught (true positives) : {tp}")
        print(f"false alarms     (false positives): {fp}")
        print(f"MISSED           (false negatives): {fn}")
        print(f"correctly cleared(true negatives) : {tn}")
        print("\nWhich of those two mistakes would hurt more in your situation?")
        print("That question decides whether you tune for precision or for recall.")


---
## Step 7 — A decision tree, and watching it overfit

A tree asks a series of yes/no questions. It is the easiest model to explain to a non-technical
person — and the easiest to break by letting it grow too deep.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

rows = []
for d in [2, 3, 4, 6, 8, 12, None]:
    Tree = DecisionTreeRegressor if task == "regression" else DecisionTreeClassifier
    t = Tree(max_depth=d, random_state=42).fit(X_train, y_train)
    tr = score(y_train, t.predict(X_train), task)
    te = score(y_test,  t.predict(X_test),  task)
    key = "R2" if task == "regression" else "Accuracy"
    rows.append({"max_depth": str(d), "TRAIN score": round(tr[key], 3), "TEST score": round(te[key], 3),
                 "gap": round(tr[key] - te[key], 3)})

depth_table = pd.DataFrame(rows)
print(depth_table.to_string(index=False))

plt.figure(figsize=(6.5, 3.4))
plt.plot(depth_table["max_depth"], depth_table["TRAIN score"], marker="o", label="train", color="#2194D4")
plt.plot(depth_table["max_depth"], depth_table["TEST score"],  marker="o", label="test",  color="#B4342F")
plt.xlabel("tree depth"); plt.ylabel("score"); plt.legend()
plt.title("Overfitting, live"); plt.tight_layout(); plt.show()


**This is the most important picture of the day.**

The train line keeps climbing — a deep enough tree memorises every training row perfectly. The test
line rises, peaks, and then *falls*. Everything to the right of that peak is the model learning noise
instead of pattern.

**Overfitting** is the gap between the two lines. Your job is to pick the depth at the peak of the
red line, not the peak of the blue one.


In [ ]:
BEST_DEPTH = 4          # <- change this to the depth at the peak of your RED line

Tree = DecisionTreeRegressor if task == "regression" else DecisionTreeClassifier
tree, s_tree = evaluate(f"DecisionTree(depth={BEST_DEPTH})", Tree(max_depth=BEST_DEPTH, random_state=42),
                        X_train, X_test, y_train, y_test, task)

print("\nWhat the tree found most useful:")
print(importances(tree, X).to_string())


### Step 7b — Read the actual rules


In [ ]:
from sklearn.tree import export_text
print(export_text(tree, feature_names=list(X.columns), max_depth=3)[:2500])


Follow one path from top to bottom and read it as a sentence: *"if X is below 12 and Y is 'Home',
predict 1."* That is a rule a person can act on — which is why trees survive in industry long after
fancier models have won on accuracy.


---
## Step 8 — The leaderboard


In [ ]:
leaderboard()


### The only three questions that matter

1. **Does anything beat the baseline?** If not, your features do not contain the answer. That is a
   legitimate finding — report it, do not fake it.
2. **By how much?** Beating the baseline by one percentage point on 80 test rows is noise.
3. **Which model would you actually hand over?** Often the simpler one. A tree you can read beats a
   forest you cannot, when the difference is small.


---
## Day 3 deliverable

In markdown at the end of your notebook:

1. **The prediction question** — in one sentence, in plain English, including whether it is regression or classification and why.
2. **The leakage paragraph** — what was excluded and why. If nothing was, say how you checked.
3. **The baseline number**, and your best model's number beside it.
4. **The overfitting table** from Step 7, and the depth you chose.
5. **The three features that mattered most**, and whether that surprised you.
6. **One sentence you would say to a non-technical person** about what your model does.

**Tonight:** try one more thing — remove your strongest feature and re-run. If the score barely
moves, that feature was not doing the work you thought it was. Bring the result tomorrow.


In [ ]:
import joblib
joblib.dump({"model": tree, "columns": list(X.columns), "task": task, "target_label": label},
            "day3_model.joblib")
X.to_csv("day3_features.csv", index=False)
pd.Series(y, name="target").to_csv("day3_target.csv", index=False)
print("Saved day3_model.joblib, day3_features.csv, day3_target.csv - bring all three tomorrow.")
